In [1]:
from pyspark import SparkContext

In [2]:
sc = SparkContext(appName="W4GA_RDD")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/19 17:01:11 INFO SparkEnv: Registering MapOutputTracker
25/10/19 17:01:11 INFO SparkEnv: Registering BlockManagerMaster
25/10/19 17:01:11 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/10/19 17:01:11 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
rdd = sc.parallelize([1,2,3,4])
rdd.count()

4

In [4]:
def hour_to_bin(hour):
    h = int(hour)
    if 0 <= h < 6:
        return "0-6"
    elif 6 <= h < 12:
        return "6-12"
    elif 12 <= h < 18:
        return "12-18"
    elif 18 <= h < 24:
        return "18-24"
    else:
        return "unknown"

In [5]:
def parse_line(line):
    parts = line.split(',', 1)
    if len(parts) != 2:
        return None
    ts = parts[1].strip()
    if ts == "" or ts.lower().startswith("timestamp"):
        return None
    try:
        time_part = ts.split(' ')[1]
        hour = int(time_part.split(':')[0])
        return (hour_to_bin(hour), 1)
    except Exception:
        return None

In [6]:
import os
_bucket = os.environ.get("GCS_BUCKET", "gs://your-bucket-name").rstrip("/")
input_path = f"{_bucket}/w4/click_file.txt"
output_path = f"{_bucket}/w4/w4_rdd_output"

In [7]:
lines = sc.textFile(input_path)

In [8]:
pairs = lines.map(parse_line).filter(lambda x: x is not None)

In [9]:
counts = pairs.reduceByKey(lambda a, b: a + b)
counts

PythonRDD[8] at RDD at PythonRDD.scala:53

In [10]:
all_bins = sc.parallelize([("0-6", 0), ("6-12", 0), ("12-18", 0), ("18-24", 0)])
merged = all_bins.union(counts).reduceByKey(lambda a, b: a + b)
merged

PythonRDD[16] at RDD at PythonRDD.scala:53

In [11]:
results = merged.collect()

In [12]:
print("Click counts by time bin:")
for b, c in sorted(results):
    print(f"{b}: {c}")

Click counts by time bin:
0-6: 44
12-18: 48
18-24: 55
6-12: 53


In [13]:
merged.map(lambda kv: f"{kv[0]},{kv[1]}") \
      .coalesce(1) \
      .saveAsTextFile(output_path)

In [ ]:
sc.stop()